# Model Evaluation

Homogeneous evaluation and analysis of the Linear, Polynomial Ridge, and Random Forest models for NiCoMn, NiFeTi, and ZnYGe.

In [1]:
import importlib
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## Configuration

In [2]:
# Resolve the project root so the notebook works inside the portable ZIP.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks" or not (PROJECT_DIR / "datasets").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "datasets"
MODEL_DIR = PROJECT_DIR / "models"
PREDICTOR_DIR = PROJECT_DIR / "predictors"

if str(PREDICTOR_DIR) not in sys.path:
    sys.path.insert(0, str(PREDICTOR_DIR))

SYSTEMS = {
    "NiCoMn": {
        "predictor": "NiCoMn_T3b_Predictor",
        "data": "NiCoMn_T3b_dataset_test.csv",
        "scaler": "NiCoMn_T3b_scaler.pkl",
        "features": ["AA_pct", "AB_pct", "BB_pct"],
    },
    "NiFeTi": {
        "predictor": "NiFeTi_T3b_Predictor",
        "data": "NiFeTi_T3b_dataset_test.csv",
        "scaler": "NiFeTi_T3b_scaler.pkl",
        "features": ["CC_pct", "AB_pct", "BC_pct"],
    },
    "ZnYGe": {
        "predictor": "ZnYGe_T3b_Predictor",
        "data": "ZnYGe_T3b_dataset_test.csv",
        "scaler": "ZnYGe_T3b_scaler.pkl",
        "features": ["CC_pct", "BB_pct", "AA_pct"],
    },
}

MODEL_TYPES = ["linear", "poly", "rf"]
MODEL_NAMES = {"linear": "Linear", "poly": "Polynomial Ridge", "rf": "Random Forest"}


## Data and evaluation

In [3]:
def inverse_scale(df, scaler_path):
    scaler = joblib.load(scaler_path)
    mean, std = scaler["mean"], scaler["std"]
    result = df.copy()
    cols = result.columns.intersection(mean.index)
    result[cols] = result[cols] * std[cols] + mean[cols]
    return result


def evaluate_system(df, predictor, features, target="Ratio"):
    predictions = {}
    rows = []

    for model_type in MODEL_TYPES:
        y_pred = np.array([
            predictor.predict_ratio(*row, model_type=model_type)
            for row in df[features].itertuples(index=False, name=None)
        ]).ravel()

        y_true = df[target].to_numpy()
        predictions[model_type] = y_pred
        rows.append({
            "Model": MODEL_NAMES[model_type],
            "R²": r2_score(y_true, y_pred),
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        })

    return pd.DataFrame(rows), predictions

## Model analysis

In [4]:
def analyze_model(predictor, features):
    linear = predictor.models["linear"]
    poly_model = predictor.models["poly"]
    poly = predictor.poly
    rf = predictor.models["rf"]

    linear_df = pd.DataFrame({
        "Feature": features,
        "Coefficient": np.asarray(linear.coef_).ravel(),
    })

    poly_df = pd.DataFrame({
        "Term": poly.get_feature_names_out(features),
        "Coefficient": np.asarray(poly_model.coef_).ravel(),
    })

    rf_estimator = rf.estimators_[0] if hasattr(rf, "estimators_") else rf
    rf_df = pd.DataFrame({
        "Feature": features,
        "Importance": rf_estimator.feature_importances_,
    }).sort_values("Importance", ascending=False).reset_index(drop=True)

    return {"linear": linear_df, "poly": poly_df, "rf": rf_df}


def prediction_analysis(df, predictions, features, model_type, target="Ratio", show_heatmap=True):
    result = df[features].copy()
    result["True Ratio"] = df[target].to_numpy()
    result["Predicted Ratio"] = np.asarray(predictions[model_type]).ravel()
    result["Error"] = result["Predicted Ratio"] - result["True Ratio"]
    result["Absolute Error"] = result["Error"].abs()

    correlations = pd.DataFrame({
        "True Ratio": result[features + ["True Ratio"]].corr()["True Ratio"].drop("True Ratio"),
        "Predicted Ratio": result[features + ["Predicted Ratio"]].corr()["Predicted Ratio"].drop("Predicted Ratio"),
    })
    correlations["Difference"] = correlations["Predicted Ratio"] - correlations["True Ratio"]

    q1, q3 = result["True Ratio"].quantile([0.25, 0.75])
    bins = [-np.inf, q1, q3, np.inf]
    labels = ["Low", "Medium", "High"]
    result["True Group"] = pd.cut(result["True Ratio"], bins=bins, labels=labels)
    result["Predicted Group"] = pd.cut(result["Predicted Ratio"], bins=bins, labels=labels)

    true_means = result.groupby("True Group", observed=False)[features].mean()
    pred_means = result.groupby("Predicted Group", observed=False)[features].mean()
    difference = pred_means - true_means

    valid = result[["True Group", "Predicted Group"]].notna().all(axis=1)
    accuracy = (result.loc[valid, "True Group"].astype(str).to_numpy() ==
                result.loc[valid, "Predicted Group"].astype(str).to_numpy()).mean()

    if show_heatmap:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        sns.heatmap(true_means, annot=True, fmt=".3f", cmap="viridis", ax=axes[0])
        axes[0].set_title("True Groups")
        sns.heatmap(pred_means, annot=True, fmt=".3f", cmap="viridis", ax=axes[1])
        axes[1].set_title(f"{MODEL_NAMES[model_type]} Predicted Groups")
        sns.heatmap(difference.abs(), annot=True, fmt=".3f", cmap="coolwarm", ax=axes[2])
        axes[2].set_title("Absolute Group Difference")
        plt.tight_layout()
        plt.show()

    metrics = {
        "R²": r2_score(result["True Ratio"], result["Predicted Ratio"]),
        "MAE": mean_absolute_error(result["True Ratio"], result["Predicted Ratio"]),
        "RMSE": np.sqrt(mean_squared_error(result["True Ratio"], result["Predicted Ratio"])),
    }

    return {
        "data": result,
        "correlations": correlations,
        "true_group_means": true_means,
        "pred_group_means": pred_means,
        "group_difference": difference,
        "metrics": metrics,
        "group_accuracy": accuracy,
    }

## Run all systems

In [8]:
all_results = {}

for system, cfg in SYSTEMS.items():
    print(f"\n{'=' * 70}\n{system}\n{'=' * 70}")

    predictor = importlib.import_module(cfg["predictor"])
    df = pd.read_csv(DATA_DIR / system / cfg["data"])
    df = inverse_scale(df, MODEL_DIR / system / cfg["scaler"])

    results, predictions = evaluate_system(df, predictor, cfg["features"])
    model_analysis = analyze_model(predictor, cfg["features"])

    print("\nModel performance")
    display(results.style.format({"R²": "{:.4f}", "MAE": "{:.4f}", "RMSE": "{:.4f}"}))

    print("\nLinear coefficients")
    display(model_analysis["linear"].style.format({"Coefficient": "{:.6f}"}))

    print("\nPolynomial coefficients")
    display(model_analysis["poly"].style.format({"Coefficient": "{:.6f}"}))

    print("\nRandom Forest feature importance")
    display(model_analysis["rf"].style.format({"Importance": "{:.4f}"}))


    all_results[system] = {
        "results": results,
        "predictions": predictions,
        "model_analysis": model_analysis,
    }


NiCoMn


d:\College\Research Intern Applications\UNSW\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
d:\College\Research Intern Applications\UNSW\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
d:\College\Research Intern Applications\UNSW\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
d:\College\Research Intern Applications\UNSW\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
d:\College\Research Intern Applications\UNSW\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but Rid


Model performance


,Model,R²,MAE,RMSE
0,Linear,0.6336,0.0571,0.0707
1,Polynomial Ridge,0.8368,0.0387,0.0472
2,Random Forest,0.7308,0.0505,0.0606



Linear coefficients


...

,Feature,Coefficient
0,CC_pct,1.194353
1,BB_pct,0.141163
2,AA_pct,-0.351015



Polynomial coefficients


,Term,Coefficient
0,1,0.000000
1,CC_pct,1.194353
2,BB_pct,0.141163
3,AA_pct,-0.351015



Random Forest feature importance


,Feature,Importance
0,CC_pct,0.9339
1,AA_pct,0.0392
2,BB_pct,0.0269


## Summary

In [6]:
summary = pd.DataFrame({
    system: all_results[system]["results"].set_index("Model")["R²"]
    for system in SYSTEMS
}).reindex(["Linear", "Polynomial Ridge", "Random Forest"])

display(summary.round(4))

,NiCoMn,NiFeTi,ZnYGe
Model,,,
Linear,0.6336,0.8404,0.8939
Polynomial Ridge,0.8368,0.8690,0.8939
Random Forest,0.7308,0.7810,0.8242
